# Simulador Interactivo — Esfera de Bloch

Este notebook permite explorar la evolución de un qubit en la esfera de Bloch aplicando puertas cuánticas de forma interactiva, **sin necesidad de Streamlit ni ninguna aplicación externa**.

## Cómo usarlo

1. Ejecuta todas las celdas (`Kernel → Restart & Run All`).
2. Usa los controles del panel interactivo para:
   - Seleccionar el estado inicial del qubit.
   - Aplicar puertas fijas (H, X, Y, Z, S, T, Sdg, Tdg).
   - Aplicar puertas paramétricas Rx, Ry, Rz, P con el slider de ángulo.
   - Ejecutar secuencias predefinidas.
   - Reiniciar o limpiar la trayectoria.

> **Requisito**: `ipywidgets` debe estar instalado y habilitado. Si no ves los controles:
> ```bash
> pip install ipywidgets
> jupyter nbextension enable --py widgetsnbextension
> ```

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..'))

import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.bloch_simulator import BlochSimulator
from src.quantum_math import QuantumMath

print('✓ Módulos cargados.')

In [ ]:
# ── Simulador y estado global ─────────────────────────────────────
sim = BlochSimulator('|0⟩')
gate_history = []


def refresh_display():
    """Actualiza la figura y el panel de información."""
    info = sim.state_info()
    alpha = info['alpha']
    beta  = info['beta']
    bv    = info['bloch_vector']

    # ── Figura Plotly ─────────────────────────────────────────────
    fig = sim.plot_trajectory(
        title=f"Esfera de Bloch  ·  {len(gate_history)} puertas aplicadas",
        dark_mode=True,
    )
    fig.update_layout(height=520, width=680)

    with plot_out:
        plot_out.clear_output(wait=True)
        fig.show()

    # ── Panel de información ──────────────────────────────────────
    sign_b = '+' if alpha.imag >= 0 else ''
    sign_c = '+' if beta.imag  >= 0 else ''
    hist_str = ' → '.join(gate_history[-12:]) if gate_history else '(ninguna)'

    info_html.value = f"""
    <div style='font-family:Inter,sans-serif; color:#e6edf3; font-size:13px;
                background:#161b22; border:1px solid #30363d;
                border-radius:10px; padding:14px 18px; line-height:2'>
      <b style='color:#58a6ff'>Estado actual</b><br>
      <span style='color:#8b949e'>|ψ⟩ =</span>
        <b>{alpha.real:.4f}{sign_b}{alpha.imag:.4f}i</b> |0⟩
        <b> + {beta.real:.4f}{sign_c}{beta.imag:.4f}i</b> |1⟩<br>
      <span style='color:#8b949e'>P(|0⟩) =</span> {info['prob_0']:.4f} &nbsp;
      <span style='color:#8b949e'>P(|1⟩) =</span> {info['prob_1']:.4f}<br>
      <span style='color:#8b949e'>Bloch (x,y,z) =</span>
        ({bv[0]:.3f}, {bv[1]:.3f}, {bv[2]:.3f})<br>
      <span style='color:#8b949e'>θ =</span> {info['theta_deg']:.2f}°&nbsp;
      <span style='color:#8b949e'>φ =</span> {info['phi_deg']:.2f}°<br>
      <br>
      <b style='color:#58a6ff'>Historial</b><br>
      <span style='color:#f0883e'>{hist_str}</span>
    </div>
    """


print('✓ Funciones de visualización definidas.')

In [ ]:
# ── Controles ipywidgets ──────────────────────────────────────────

# Estilo común para los botones
BTN_STYLE  = dict(button_color='#1f6feb', font_weight='bold')
BTN_LAYOUT = widgets.Layout(width='100%', margin='3px 0')

# ── 1) Estado inicial ─────────────────────────────────────────────
state_dd = widgets.Dropdown(
    options=list(BlochSimulator.INITIAL_STATES.keys()),
    description='Inicio:',
    style={'description_width': '60px'},
    layout=widgets.Layout(width='100%'),
)
btn_reset = widgets.Button(
    description='🔄 Reiniciar',
    style=BTN_STYLE, layout=BTN_LAYOUT,
)

# ── 2) Puertas fijas ──────────────────────────────────────────────
gate_dd = widgets.Dropdown(
    options=['H', 'X', 'Y', 'Z', 'S', 'T', 'Sdg', 'Tdg', 'I'],
    description='Puerta:',
    style={'description_width': '60px'},
    layout=widgets.Layout(width='100%'),
)
btn_gate = widgets.Button(
    description='Aplicar puerta fija',
    style=BTN_STYLE, layout=BTN_LAYOUT,
)

# ── 3) Puertas paramétricas ───────────────────────────────────────
param_dd = widgets.Dropdown(
    options=['Rx', 'Ry', 'Rz', 'P'],
    description='Puerta:',
    style={'description_width': '60px'},
    layout=widgets.Layout(width='100%'),
)
angle_slider = widgets.IntSlider(
    value=90, min=-360, max=360, step=5,
    description='Ángulo (°):',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='100%'),
)
btn_param = widgets.Button(
    description='Aplicar puerta paramétrica',
    style=BTN_STYLE, layout=BTN_LAYOUT,
)

# ── 4) Secuencias predefinidas ────────────────────────────────────
SEQUENCES = {
    'Hadamard simple (H)': [{'gate': 'H'}],
    'Superposición + fase (H → S)': [{'gate': 'H'}, {'gate': 'S'}],
    'Teleportación (H → Z → H)': [
        {'gate': 'H'}, {'gate': 'Z'}, {'gate': 'H'}],
    'Rotación X completa (4×Rx45°)': [
        {'gate': 'Rx', 'theta': np.pi/4}] * 4,
    'T⁸ = I': [{'gate': 'T'}] * 8,
    'Tour Bloch (H → S → T → H)': [
        {'gate': 'H'}, {'gate': 'S'}, {'gate': 'T'}, {'gate': 'H'}],
}
seq_dd = widgets.Dropdown(
    options=list(SEQUENCES.keys()),
    description='Secuencia:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='100%'),
)
btn_seq = widgets.Button(
    description='▶ Aplicar secuencia',
    style=BTN_STYLE, layout=BTN_LAYOUT,
)

# ── 5) Limpiar trayectoria ────────────────────────────────────────
btn_clear = widgets.Button(
    description='🗑️ Limpiar trayectoria',
    style=dict(button_color='#6e7681', font_weight='bold'),
    layout=BTN_LAYOUT,
)

# ── Salidas ───────────────────────────────────────────────────────
plot_out  = widgets.Output()
info_html = widgets.HTML()


# ── Callbacks ─────────────────────────────────────────────────────
def on_reset(_):
    global gate_history
    sim.set_initial_state(state_dd.value)
    gate_history = []
    refresh_display()

def on_gate(_):
    g = gate_dd.value
    sim.apply_gate(g)
    gate_history.append(g)
    refresh_display()

def on_param(_):
    g   = param_dd.value
    ang = np.radians(angle_slider.value)
    label_ang = f'{angle_slider.value}°'
    if g == 'P':
        sim.apply_gate(g, phi=ang)
    else:
        sim.apply_gate(g, theta=ang)
    gate_history.append(f'{g}({label_ang})')
    refresh_display()

def on_seq(_):
    for step in SEQUENCES[seq_dd.value]:
        g = step['gate']
        kw = {k: v for k, v in step.items() if k != 'gate'}
        sim.apply_gate(g, **kw)
        gate_history.append(g)
    refresh_display()

def on_clear(_):
    current = sim.state
    sim.set_initial_state('|0⟩')
    sim._state = current
    bv = QuantumMath.bloch_vector(current)
    sim._trajectory = [bv]
    sim._gate_labels = []
    gate_history.clear()
    refresh_display()

btn_reset.on_click(on_reset)
btn_gate.on_click(on_gate)
btn_param.on_click(on_param)
btn_seq.on_click(on_seq)
btn_clear.on_click(on_clear)


# ── Paneles de control ────────────────────────────────────────────
def section(title, *children):
    header = widgets.HTML(
        f"<b style='color:#58a6ff;font-family:Inter,sans-serif;'"
        f">{title}</b>"
    )
    return widgets.VBox([header, *children],
                        layout=widgets.Layout(
                            border='1px solid #30363d', border_radius='8px',
                            padding='10px', margin='6px 0',
                            background_color='#161b22'))

sidebar = widgets.VBox([
    section('Estado inicial', state_dd, btn_reset),
    section('Puerta fija', gate_dd, btn_gate),
    section('Puerta paramétrica', param_dd, angle_slider, btn_param),
    section('Secuencias', seq_dd, btn_seq),
    btn_clear,
], layout=widgets.Layout(width='300px', padding='6px'))

right_panel = widgets.VBox(
    [plot_out, info_html],
    layout=widgets.Layout(flex='1', padding='6px'),
)

ui = widgets.HBox(
    [sidebar, right_panel],
    layout=widgets.Layout(
        background_color='#0d1117',
        border='1px solid #21262d',
        border_radius='12px',
    ),
)

# Primer renderizado
display(ui)
refresh_display()

## Sección educativa — Referencia rápida

### Estados básicos

| Estado | α | β | Posición en Bloch |
|--------|---|---|-------------------|
| `\|0⟩` | 1 | 0 | Polo norte (0,0,1) |
| `\|1⟩` | 0 | 1 | Polo sur (0,0,−1) |
| `\|+⟩` | 1/√2 | 1/√2 | Ecuador (+x) |
| `\|-⟩` | 1/√2 | −1/√2 | Ecuador (−x) |
| `\|i⟩` | 1/√2 | i/√2 | Ecuador (+y) |
| `\|−i⟩` | 1/√2 | −i/√2 | Ecuador (−y) |

### Puertas de 1 qubit

| Puerta | Efecto en Bloch | Matriz |
|--------|-----------------|--------|
| H | Hadamard: intercambia X↔Z | [[1,1],[1,−1]]/√2 |
| X | Rotación π en X (NOT) | [[0,1],[1,0]] |
| Y | Rotación π en Y | [[0,−i],[i,0]] |
| Z | Rotación π en Z (inversión de fase) | [[1,0],[0,−1]] |
| S | Rotación π/2 en Z | [[1,0],[0,i]] |
| T | Rotación π/4 en Z | [[1,0],[0,e^{iπ/4}]] |
| Rx(θ) | Rotación θ radianes en X | cos(θ/2)I − i sin(θ/2)X |
| Ry(θ) | Rotación θ radianes en Y | cos(θ/2)I − i sin(θ/2)Y |
| Rz(θ) | Rotación θ radianes en Z | cos(θ/2)I − i sin(θ/2)Z |

### Fórmulas del vector de Bloch

El estado general de un qubit puro se parametriza como:

$$|\psi\rangle = \cos\!\left(\frac{\theta}{2}\right)|0\rangle + e^{i\varphi}\sin\!\left(\frac{\theta}{2}\right)|1\rangle$$

con el vector de Bloch:

$$\vec{r} = (\sin\theta\cos\varphi,\; \sin\theta\sin\varphi,\; \cos\theta)$$

In [ ]:
# ── Exploración libre: experimento personalizado ──────────────────
# Modifica las líneas siguientes y ejecuta la celda.

exp = BlochSimulator('|0⟩')

exp.apply_sequence([
    {'gate': 'H'},
    {'gate': 'Rx', 'theta': np.pi / 3},
    {'gate': 'S'},
    {'gate': 'T'},
    {'gate': 'H'},
])

info = exp.state_info()
print(f"Estado final:")
print(f"  α = {info['alpha']:.4f}")
print(f"  β = {info['beta']:.4f}")
print(f"  P(|0⟩) = {info['prob_0']:.4f}")
print(f"  Bloch  = {tuple(round(c, 3) for c in info['bloch_vector'])}")

fig_exp = exp.plot_trajectory(title='Experimento personalizado', dark_mode=True)
fig_exp.update_layout(height=480, width=700)
fig_exp.show()